# Cell 1 — Setup + find swapped dataset

In [1]:
!pip install -q ultralytics huggingface_hub scikit-learn opencv-python pyyaml

from pathlib import Path
import shutil
import yaml
import pandas as pd
import numpy as np
import torch
import cv2
from collections import Counter, defaultdict
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# Find YOLO dataset
candidate_dirs = []

for yaml_path in INPUT_ROOT.rglob("data.yaml"):
    folder = yaml_path.parent
    
    if not (folder / "images" / "train").exists():
        continue
    if not (folder / "labels" / "train").exists():
        continue
    if not (folder / "images" / "val").exists():
        continue
    if not (folder / "labels" / "val").exists():
        continue
    if not (folder / "images" / "test").exists():
        continue
    if not (folder / "labels" / "test").exists():
        continue

    score = 0
    names_list = []

    try:
        cfg = yaml.safe_load(yaml_path.read_text())
        names = cfg.get("names", [])
        names_list = list(names.values()) if isinstance(names, dict) else list(names)
        joined = " ".join([str(x).lower() for x in names_list])

        if "player_team1" in joined:
            score += 10
        if "player_team2" in joined:
            score += 10
        if "ball" in joined:
            score += 5
        if "referee" in joined:
            score += 5
        if "black" in str(folder).lower() or "swapped" in str(folder).lower():
            score += 3

    except Exception:
        pass

    candidate_dirs.append((score, folder, names_list))

if len(candidate_dirs) == 0:
    raise FileNotFoundError("Could not find YOLO dataset in /kaggle/input")

candidate_dirs = sorted(candidate_dirs, key=lambda x: x[0], reverse=True)

original_dataset_dir = candidate_dirs[0][1]

print("\nDatasets found:")
for i, (score, folder, names) in enumerate(candidate_dirs):
    print(i, "| score:", score, "|", folder, "|", names)

print("\nUsing dataset:", original_dataset_dir)

# =========================================================
# Robust helpers for Kaggle restarts / reruns
# =========================================================

def find_model_path(prefer="best", raise_error=True):
    """
    Locate the trained YOLO model in /kaggle/working.
    Prefer best.pt, then last.pt. This is safer than relying only on
    notebook variables, which disappear after a Kaggle/kernel restart.
    """
    search_roots = []
    if "PROJECT_DIR" in globals():
        search_roots.append(Path(PROJECT_DIR))
    search_roots.append(WORK_DIR)

    variable_candidates = []
    for var_name in ["BEST_MODEL_PATH", "LAST_MODEL_PATH"] if prefer == "best" else ["LAST_MODEL_PATH", "BEST_MODEL_PATH"]:
        if var_name in globals():
            try:
                p = Path(globals()[var_name])
                if p.exists():
                    variable_candidates.append(p)
            except Exception:
                pass

    pattern_order = ["**/weights/best.pt", "**/best.pt", "**/weights/last.pt", "**/last.pt"]
    if prefer == "last":
        pattern_order = ["**/weights/last.pt", "**/last.pt", "**/weights/best.pt", "**/best.pt"]

    candidates = list(variable_candidates)
    for root in search_roots:
        if root.exists():
            for pattern in pattern_order:
                candidates.extend(root.rglob(pattern))

    # Keep unique existing candidates and prefer newest file.
    unique = []
    seen = set()
    for p in candidates:
        p = Path(p)
        if p.exists() and p not in seen:
            unique.append(p)
            seen.add(p)

    if unique:
        unique = sorted(unique, key=lambda p: p.stat().st_mtime, reverse=True)
        return unique[0]

    if raise_error:
        raise RuntimeError(
            "No trained model was found in /kaggle/working. "
            "This usually means the Kaggle session/kernel restarted after being idle, "
            "so temporary outputs were deleted. Run the notebook from the top in one session, "
            "or rerun the training cell before validation/export."
        )
    return None


def safe_export_outputs(stage="final", cleanup_heavy=False, require_model=False):
    """
    Create a small zip with the model, CSV results, and important plots.
    Also works as a checkpoint export after training/validation so results
    are available even if a later analysis cell fails.
    """
    export_name = "final_swapped_3class_outputs" if stage == "final" else f"checkpoint_{stage}_swapped_3class_outputs"
    final_dir = WORK_DIR / export_name

    if final_dir.exists():
        shutil.rmtree(final_dir)
    final_dir.mkdir(parents=True, exist_ok=True)

    print(f"Creating export folder: {final_dir}")

    copied_any = False
    copied_model = False

    # Copy best/last model if present.
    best_path = find_model_path("best", raise_error=False)
    last_path = find_model_path("last", raise_error=False)

    if best_path is not None:
        shutil.copy2(best_path, final_dir / "best.pt")
        print("Copied best model:", best_path)
        copied_any = True
        copied_model = True

    if last_path is not None and last_path != best_path:
        shutil.copy2(last_path, final_dir / "last.pt")
        print("Copied last model:", last_path)
        copied_any = True
        copied_model = True

    if require_model and not copied_model:
        raise RuntimeError(
            "Could not export because no best.pt or last.pt exists in /kaggle/working. "
            "Your Kaggle session most likely restarted or the training cell did not finish. "
            "Run all cells from the top without leaving the session idle, then run this export cell."
        )

    # Copy CSV results from anywhere under /kaggle/working.
    csv_patterns = [
        "three_class_swapped_per_class.csv",
        "three_class_swapped_summary.csv",
        "swapped_error_analysis_summary.csv",
        "swapped_threshold_comparison.csv",
        "swapped_image_error_analysis.csv",
    ]

    for pattern in csv_patterns:
        for p in WORK_DIR.rglob(pattern):
            if final_dir in p.parents:
                continue
            target = final_dir / p.name
            shutil.copy2(p, target)
            print("Copied CSV:", p)
            copied_any = True

    # Copy important training/evaluation plots only.
    important_plot_names = {
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
        "PR_curve.png",
    }

    for p in WORK_DIR.rglob("*.png"):
        if final_dir in p.parents:
            continue
        if p.name in important_plot_names:
            target_name = p.parent.name + "_" + p.name
            shutil.copy2(p, final_dir / target_name)
            print("Copied plot:", p)
            copied_any = True

    readme_text = f"""
YOLOv8 3-class football model outputs
Export stage: {stage}

Classes:
0 = player
1 = ball
2 = referee

Expected important files:
- best.pt / last.pt
- three_class_swapped_per_class.csv
- three_class_swapped_summary.csv
- swapped_error_analysis_summary.csv
- swapped_threshold_comparison.csv
- swapped_image_error_analysis.csv

Note:
If model files are missing, Kaggle probably restarted the session and deleted /kaggle/working.
Run all cells from the top in one session, or rerun training before export.
"""
    (final_dir / "README.txt").write_text(readme_text)

    zip_path = shutil.make_archive(
        base_name=str(WORK_DIR / export_name),
        format="zip",
        root_dir=str(final_dir)
    )

    print("Created zip:", zip_path)

    if cleanup_heavy:
        heavy_paths = [
            WORK_DIR / "football_yolo_dataset_3cls_swapped",
            WORK_DIR / "three_class_swapped_experiments",
            WORK_DIR / "runs",
        ]

        for path in heavy_paths:
            if path.exists():
                print("Deleting heavy folder:", path)
                shutil.rmtree(path, ignore_errors=True)

        for cache_file in WORK_DIR.rglob("*.cache"):
            try:
                cache_file.unlink()
                print("Deleted cache:", cache_file)
            except Exception:
                pass

        gc.collect()

    print("\nFiles in export folder:")
    for p in sorted(final_dir.iterdir()):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"{p.name} - {size_mb:.2f} MB")

    if not copied_any:
        print("\nWARNING: No model/results were found. This zip only contains README.txt.")

    return zip_path




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires nu

# Cell 2 — Verify split

In [2]:
import re
from collections import Counter

def get_video_name(stem):
    if "_frame_" in stem:
        return stem.split("_frame_")[0]
    return "unknown"

def extract_frame_number(path):
    match = re.search(r"_frame_(\d+)", path.stem)
    if match:
        return int(match.group(1))
    return -1

def count_videos(images_dir):
    counts = Counter()
    frames = defaultdict(list)

    for p in images_dir.glob("*.jpg"):
        video = get_video_name(p.stem)
        frame = extract_frame_number(p)
        counts[video] += 1
        frames[video].append(frame)

    return counts, frames

train_counts, train_frames = count_videos(original_dataset_dir / "images" / "train")
val_counts, val_frames = count_videos(original_dataset_dir / "images" / "val")
test_counts, test_frames = count_videos(original_dataset_dir / "images" / "test")

print("TRAIN videos:")
for video, count in train_counts.most_common():
    fs = train_frames[video]
    print(video, "| count:", count, "| min:", min(fs), "| max:", max(fs))

print("\nVAL videos:")
for video, count in val_counts.most_common():
    fs = val_frames[video]
    print(video, "| count:", count, "| min:", min(fs), "| max:", max(fs))

print("\nTEST videos:")
for video, count in test_counts.most_common():
    fs = test_frames[video]
    print(video, "| count:", count, "| min:", min(fs), "| max:", max(fs))

def count_label_classes(labels_dir):
    counter = Counter()

    for label_path in labels_dir.glob("*.txt"):
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) >= 1:
                counter[int(float(parts[0]))] += 1

    return counter

print("\nOriginal TRAIN label counts:", count_label_classes(original_dataset_dir / "labels" / "train"))
print("Original VAL label counts:", count_label_classes(original_dataset_dir / "labels" / "val"))
print("Original TEST label counts:", count_label_classes(original_dataset_dir / "labels" / "test"))

TRAIN videos:
lightblue_green_white | count: 1057 | min: 0 | max: 5280
Green_and_Blue | count: 720 | min: 0 | max: 3595
white_and_yellow | count: 720 | min: 0 | max: 3595
hashemmatch02 | count: 720 | min: 0 | max: 3595
black_and_red | count: 703 | min: 180 | max: 3690
Green_and_Blue_part_2 | count: 360 | min: 0 | max: 1795

VAL videos:
white_and_black_2_mins | count: 742 | min: 0 | max: 3705
hashemmatch01 | count: 720 | min: 0 | max: 3595

TEST videos:
lightblue_red | count: 720 | min: 0 | max: 3595

Original TRAIN label counts: Counter({1: 21859, 0: 21383, 3: 4925, 2: 3542})
Original VAL label counts: Counter({1: 8472, 0: 7665, 3: 1614, 2: 1414})
Original TEST label counts: Counter({1: 3158, 0: 3115, 3: 787, 2: 641})


# Cell 3 — Convert 4 classes

In [3]:
# =========================================================
# Convert 4-class dataset to 3-class dataset
# =========================================================

CONVERTED_DIR = WORK_DIR / "football_yolo_dataset_3cls_swapped"

if CONVERTED_DIR.exists():
    shutil.rmtree(CONVERTED_DIR)

for split in ["train", "val", "test"]:
    (CONVERTED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (CONVERTED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Old classes:
# 0 player_team1 -> 0 player
# 1 player_team2 -> 0 player
# 2 ball         -> 1 ball
# 3 referee      -> 2 referee

old_to_new = {
    0: 0,
    1: 0,
    2: 1,
    3: 2
}

def convert_label_file(src_label_path, dst_label_path):
    new_lines = []

    if not src_label_path.exists():
        dst_label_path.write_text("")
        return Counter()

    class_counter = Counter()

    for line in src_label_path.read_text().splitlines():
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        old_cls = int(float(parts[0]))

        if old_cls not in old_to_new:
            continue

        new_cls = old_to_new[old_cls]
        new_line = " ".join([str(new_cls)] + parts[1:])
        new_lines.append(new_line)

        class_counter[new_cls] += 1

    dst_label_path.write_text("\n".join(new_lines))

    return class_counter

overall_counts = {
    "train": Counter(),
    "val": Counter(),
    "test": Counter()
}

for split in ["train", "val", "test"]:
    src_img_dir = original_dataset_dir / "images" / split
    src_lbl_dir = original_dataset_dir / "labels" / split

    dst_img_dir = CONVERTED_DIR / "images" / split
    dst_lbl_dir = CONVERTED_DIR / "labels" / split

    image_paths = sorted(list(src_img_dir.glob("*.jpg")))

    print(f"\nConverting {split}: {len(image_paths)} images")

    for img_path in image_paths:
        shutil.copy2(img_path, dst_img_dir / img_path.name)

        src_label_path = src_lbl_dir / f"{img_path.stem}.txt"
        dst_label_path = dst_lbl_dir / f"{img_path.stem}.txt"

        counts = convert_label_file(src_label_path, dst_label_path)
        overall_counts[split].update(counts)

print("\nConverted label counts:")
for split in ["train", "val", "test"]:
    print(split, overall_counts[split])

data_yaml_3cls = CONVERTED_DIR / "data.yaml"

data_cfg = {
    "path": str(CONVERTED_DIR),
    "train": "train_weighted.txt",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "player",
        1: "ball",
        2: "referee"
    }
}

with open(data_yaml_3cls, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("\nCreated:", data_yaml_3cls)
print("Converted dataset:", CONVERTED_DIR)


Converting train: 4280 images

Converting val: 1462 images

Converting test: 720 images

Converted label counts:
train Counter({0: 43242, 2: 4925, 1: 3542})
val Counter({0: 16137, 2: 1614, 1: 1414})
test Counter({0: 6273, 2: 787, 1: 641})

Created: /kaggle/working/football_yolo_dataset_3cls_swapped/data.yaml
Converted dataset: /kaggle/working/football_yolo_dataset_3cls_swapped


# Cell 4 — Create weighted train list

In [4]:
# =========================================================
# Create weighted train list
# =========================================================

TRAIN_IMAGES_DIR = CONVERTED_DIR / "images" / "train"
TRAIN_LABELS_DIR = CONVERTED_DIR / "labels" / "train"

train_images = sorted(TRAIN_IMAGES_DIR.glob("*.jpg"))

weighted_train_list = []

for img_path in train_images:
    label_path = TRAIN_LABELS_DIR / f"{img_path.stem}.txt"

    has_ball = False
    has_referee = False

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 1:
                continue

            cls_id = int(float(parts[0]))

            if cls_id == 1:
                has_ball = True
            elif cls_id == 2:
                has_referee = True

    repeat = 1

    # Light oversampling only
    if has_ball:
        repeat += 1

    if has_referee:
        repeat += 1

    for _ in range(repeat):
        weighted_train_list.append(str(img_path))

train_weighted_path = CONVERTED_DIR / "train_weighted.txt"

with open(train_weighted_path, "w") as f:
    f.write("\n".join(weighted_train_list))

print("Original train images:", len(train_images))
print("Weighted train entries:", len(weighted_train_list))
print("Saved:", train_weighted_path)

Original train images: 4280
Weighted train entries: 10858
Saved: /kaggle/working/football_yolo_dataset_3cls_swapped/train_weighted.txt


# Cell 5 — Train 3-class model from Hugging Face

In [5]:
# =========================================================
# Train 3-class model from Hugging Face football model
# Robust version: can skip completed training or resume if last.pt exists
# =========================================================

PROJECT_DIR = WORK_DIR / "three_class_swapped_experiments"
exp_name = "hf_football_3cls_swapped_960"
EXP_DIR = PROJECT_DIR / exp_name
WEIGHTS_DIR = EXP_DIR / "weights"
BEST_MODEL_PATH = WEIGHTS_DIR / "best.pt"
LAST_MODEL_PATH = WEIGHTS_DIR / "last.pt"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# If training already completed in this same Kaggle session, do not waste time.
if BEST_MODEL_PATH.exists():
    print("Training output already exists, skipping training:", BEST_MODEL_PATH)

else:
    hf_model_path = hf_hub_download(
        repo_id="uisikdag/yolo-v8-football-players-detection",
        filename="best.pt"
    )

    print("HF model path:", hf_model_path)

    # If a previous interrupted run has last.pt, resume it.
    # This only works while /kaggle/working still exists; if Kaggle restarted,
    # the checkpoint is gone and training starts from the HF model again.
    if LAST_MODEL_PATH.exists():
        print("Found interrupted checkpoint. Resuming from:", LAST_MODEL_PATH)
        model = YOLO(str(LAST_MODEL_PATH))
        train_results = model.train(resume=True)
    else:
        model = YOLO(hf_model_path)

        train_results = model.train(
            data=str(data_yaml_3cls),
            epochs=80,
            imgsz=960,
            batch=8,
            device=0,
            project=str(PROJECT_DIR),
            name=exp_name,
            exist_ok=True,          # IMPORTANT: keep path stable on reruns
            patience=20,

            optimizer="AdamW",
            lr0=0.001,
            lrf=0.01,
            cos_lr=True,

            hsv_h=0.015,
            hsv_s=0.40,
            hsv_v=0.30,
            degrees=2.0,
            translate=0.10,
            scale=0.50,
            shear=0.0,
            perspective=0.0,
            fliplr=0.50,
            mosaic=0.40,
            mixup=0.03,
            close_mosaic=15,

            cache=False,
            workers=2,
            save_period=1,          # IMPORTANT: save epoch checkpoints for safer reruns
            verbose=True
        )

# Re-locate paths safely in case Ultralytics changed anything.
BEST_MODEL_PATH = find_model_path("best")
LAST_MODEL_PATH = find_model_path("last", raise_error=False) or BEST_MODEL_PATH

print("Best model:", BEST_MODEL_PATH)
print("Last model:", LAST_MODEL_PATH)

# Create a checkpoint zip immediately after training.
# This gives you a downloadable model even if a later evaluation cell fails.
safe_export_outputs(stage="after_training", cleanup_heavy=False, require_model=True)


best.pt:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

HF model path: /root/.cache/huggingface/hub/models--uisikdag--yolo-v8-football-players-detection/snapshots/01c4d0e18813ac75a2c73cc35145bc240af85342/best.pt
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/football_yolo_dataset_3cls_swapped/data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.03,

'/kaggle/working/checkpoint_after_training_swapped_3class_outputs.zip'

# Cell 6 — YOLO validation on test

In [6]:
# =========================================================
# Validate best model on test set
# =========================================================

BEST_MODEL_PATH = find_model_path("best")
best_model = YOLO(str(BEST_MODEL_PATH))

metrics = best_model.val(
    data=str(data_yaml_3cls),
    split="test",
    imgsz=960,
    batch=8,
    device=0,
    conf=0.001,
    iou=0.6,
    plots=True,
    save_json=False,
    project=str(PROJECT_DIR),
    name=f"{exp_name}_test_eval"
)

names = ["player", "ball", "referee"]

per_class_rows = []

for i, class_name in enumerate(names):
    per_class_rows.append({
        "experiment": exp_name,
        "class": class_name,
        "precision": float(metrics.box.p[i]) if i < len(metrics.box.p) else None,
        "recall": float(metrics.box.r[i]) if i < len(metrics.box.r) else None,
        "mAP50": float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else None,
        "mAP50_95": float(metrics.box.ap[i]) if i < len(metrics.box.ap) else None,
    })

per_class_df = pd.DataFrame(per_class_rows)

summary_df = pd.DataFrame([{
    "experiment": exp_name,
    "model": str(BEST_MODEL_PATH),
    "dataset": str(CONVERTED_DIR),
    "imgsz": 960,
    "epochs": 80,
    "batch": 8,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision_mean": float(np.mean(metrics.box.p)),
    "recall_mean": float(np.mean(metrics.box.r)),
}])

per_class_path = WORK_DIR / "three_class_swapped_per_class.csv"
summary_path = WORK_DIR / "three_class_swapped_summary.csv"

per_class_df.to_csv(per_class_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Summary:")
display(summary_df)

print("Per-class:")
display(per_class_df)

print("Saved:")
print(per_class_path)
print(summary_path)

# Save a checkpoint zip after validation metrics are written.
safe_export_outputs(stage="after_test_validation", cleanup_heavy=False, require_model=True)


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1473.1±529.3 MB/s, size: 65.6 KB)
val: Scanning /kaggle/working/football_yolo_dataset_3cls_swapped/labels/test... 720 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 720/720 1.2Kit/s 0.6s<0.1s
val: New cache created: /kaggle/working/football_yolo_dataset_3cls_swapped/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 90/90 6.9it/s 13.1s0.1s
                   all        720       7701      0.885      0.811      0.846       0.42
                player        718       6273       0.93      0.972      0.972      0.568
                  ball        626        641      0.813      0.571      0.637       0.25
               referee        431        787      0.913      0.889      0.928      0.442
Speed: 

,experiment,model,dataset,imgsz,epochs,batch,mAP50,mAP50_95,precision_mean,recall_mean
0,hf_football_3cls_swapped_960,/kaggle/working/three_class_swapped_experiment...,/kaggle/working/football_yolo_dataset_3cls_swa...,960,80,8,0.845881,0.42015,0.885452,0.810846


Per-class:


,experiment,class,precision,recall,mAP50,mAP50_95
0,hf_football_3cls_swapped_960,player,0.930479,0.972103,0.972415,0.568009
1,hf_football_3cls_swapped_960,ball,0.813142,0.570983,0.636839,0.250167
2,hf_football_3cls_swapped_960,referee,0.912734,0.889454,0.928388,0.442275


Saved:
/kaggle/working/three_class_swapped_per_class.csv
/kaggle/working/three_class_swapped_summary.csv
Creating export folder: /kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs
Copied best model: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960/weights/best.pt
Copied CSV: /kaggle/working/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/three_class_swapped_summary.csv
Copied plot: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960_test_eval/confusion_matrix_normalized.png
Copied plot: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960_test_eval/confusion_matrix.png
Copied plot: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960/confusion_matrix_normalized.png
Copied plot: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960/confusion_matrix.png
Copied plot: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_s

'/kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs.zip'

# Cell 7 — Error analysis: player/referee confusion + team recall

In [7]:
# =========================================================
# Custom evaluation on original 4-class labels
# Measures:
# - Team A recall
# - Team B recall
# - Referee recall
# - Referee wrongly as player
# - Players wrongly as referee
# =========================================================

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50
IMG_SIZE = 960

ORIGINAL_TEST_IMAGES = original_dataset_dir / "images" / "test"
ORIGINAL_TEST_LABELS = original_dataset_dir / "labels" / "test"

BEST_MODEL_PATH = find_model_path("best")
eval_model = YOLO(str(BEST_MODEL_PATH))

def yolo_to_xyxy(xc, yc, bw, bh, img_w, img_h):
    x1 = (xc - bw / 2) * img_w
    y1 = (yc - bh / 2) * img_h
    x2 = (xc + bw / 2) * img_w
    y2 = (yc + bh / 2) * img_h
    return [float(x1), float(y1), float(x2), float(y2)]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)

    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0

    return inter_area / union

def greedy_match(preds, gts, iou_threshold=0.5):
    candidates = []

    for pi, p in enumerate(preds):
        for gi, g in enumerate(gts):
            iou = box_iou(p["box"], g["box"])

            if iou >= iou_threshold:
                candidates.append((iou, pi, gi))

    candidates.sort(reverse=True)

    matched_preds = set()
    matched_gts = set()
    matches = []

    for iou, pi, gi in candidates:
        if pi in matched_preds or gi in matched_gts:
            continue

        matched_preds.add(pi)
        matched_gts.add(gi)

        matches.append({
            "pred_idx": pi,
            "gt_idx": gi,
            "iou": float(iou)
        })

    return matches

def load_original_labels(label_path, img_w, img_h):
    objects = []

    if not label_path.exists():
        return objects

    for line in label_path.read_text().splitlines():
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        cls_id = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:])

        box = yolo_to_xyxy(xc, yc, bw, bh, img_w, img_h)

        item = {
            "old_cls": cls_id,
            "box": box
        }

        if cls_id == 0:
            item["type"] = "player"
            item["team":] = 0

        elif cls_id == 1:
            item["type"] = "player"
            item["team"] = 1

        elif cls_id == 2:
            item["type"] = "ball"
            item["team"] = None

        elif cls_id == 3:
            item["type"] = "referee"
            item["team"] = None

        else:
            item["type"] = "unknown"
            item["team"] = None

        # Fix possible typo-safe team assignment
        if cls_id == 0:
            item["team"] = 0

        objects.append(item)

    return objects

counts = Counter()
image_rows = []

image_paths = sorted(ORIGINAL_TEST_IMAGES.glob("*.jpg"))

print("Test images:", len(image_paths))

for idx, img_path in enumerate(image_paths):
    if idx % 100 == 0:
        print(f"Processing {idx}/{len(image_paths)}")

    frame = cv2.imread(str(img_path))

    if frame is None:
        continue

    h, w = frame.shape[:2]

    gt_objects = load_original_labels(
        ORIGINAL_TEST_LABELS / f"{img_path.stem}.txt",
        w,
        h
    )

    gt_players = [g for g in gt_objects if g["type"] == "player"]
    gt_team_a = [g for g in gt_players if g["team"] == 0]
    gt_team_b = [g for g in gt_players if g["team"] == 1]
    gt_refs = [g for g in gt_objects if g["type"] == "referee"]
    gt_balls = [g for g in gt_objects if g["type"] == "ball"]

    result = eval_model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD,
        device=0,
        verbose=False
    )[0]

    pred_players = []
    pred_balls = []
    pred_refs = []

    if result.boxes is not None:
        xyxy = result.boxes.xyxy.cpu().numpy()
        clss = result.boxes.cls.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()

        for box, cls_id, conf in zip(xyxy, clss, confs):
            pred = {
                "box": [float(x) for x in box],
                "cls": int(cls_id),
                "conf": float(conf)
            }

            if cls_id == 0:
                pred_players.append(pred)
            elif cls_id == 1:
                pred_balls.append(pred)
            elif cls_id == 2:
                pred_refs.append(pred)

    counts["gt_team_a"] += len(gt_team_a)
    counts["gt_team_b"] += len(gt_team_b)
    counts["gt_players"] += len(gt_players)
    counts["gt_referees"] += len(gt_refs)
    counts["gt_balls"] += len(gt_balls)

    counts["pred_players"] += len(pred_players)
    counts["pred_referees"] += len(pred_refs)
    counts["pred_balls"] += len(pred_balls)

    team_a_matches = greedy_match(pred_players, gt_team_a, IOU_THRESHOLD)
    team_b_matches = greedy_match(pred_players, gt_team_b, IOU_THRESHOLD)
    ref_as_ref_matches = greedy_match(pred_refs, gt_refs, IOU_THRESHOLD)
    ref_as_player_matches = greedy_match(pred_players, gt_refs, IOU_THRESHOLD)
    player_as_ref_matches = greedy_match(pred_refs, gt_players, IOU_THRESHOLD)
    ball_matches = greedy_match(pred_balls, gt_balls, IOU_THRESHOLD)

    counts["matched_team_a_as_player"] += len(team_a_matches)
    counts["matched_team_b_as_player"] += len(team_b_matches)
    counts["matched_referee_as_referee"] += len(ref_as_ref_matches)
    counts["referee_wrongly_as_player"] += len(ref_as_player_matches)
    counts["player_wrongly_as_referee"] += len(player_as_ref_matches)
    counts["matched_balls"] += len(ball_matches)

    image_rows.append({
        "image": img_path.name,
        "gt_team_a": len(gt_team_a),
        "gt_team_b": len(gt_team_b),
        "gt_referees": len(gt_refs),
        "gt_balls": len(gt_balls),
        "pred_players": len(pred_players),
        "pred_referees": len(pred_refs),
        "pred_balls": len(pred_balls),
        "team_a_matched_as_player": len(team_a_matches),
        "team_b_matched_as_player": len(team_b_matches),
        "referee_as_referee": len(ref_as_ref_matches),
        "referee_wrongly_as_player": len(ref_as_player_matches),
        "player_wrongly_as_referee": len(player_as_ref_matches),
        "ball_matched": len(ball_matches),
    })

overall_error_summary = pd.DataFrame([{
    "conf_threshold": CONF_THRESHOLD,
    "gt_team_a": counts["gt_team_a"],
    "gt_team_b": counts["gt_team_b"],
    "gt_players": counts["gt_players"],
    "gt_referees": counts["gt_referees"],
    "gt_balls": counts["gt_balls"],

    "team_a_detection_recall": round(counts["matched_team_a_as_player"] / counts["gt_team_a"], 4) if counts["gt_team_a"] > 0 else 0,
    "team_b_detection_recall": round(counts["matched_team_b_as_player"] / counts["gt_team_b"], 4) if counts["gt_team_b"] > 0 else 0,
    "player_detection_recall": round((counts["matched_team_a_as_player"] + counts["matched_team_b_as_player"]) / counts["gt_players"], 4) if counts["gt_players"] > 0 else 0,
    "ball_recall": round(counts["matched_balls"] / counts["gt_balls"], 4) if counts["gt_balls"] > 0 else 0,
    "referee_recall": round(counts["matched_referee_as_referee"] / counts["gt_referees"], 4) if counts["gt_referees"] > 0 else 0,

    "referee_wrong_as_player_rate": round(counts["referee_wrongly_as_player"] / counts["gt_referees"], 4) if counts["gt_referees"] > 0 else 0,
    "player_wrong_as_referee_rate": round(counts["player_wrongly_as_referee"] / counts["gt_players"], 4) if counts["gt_players"] > 0 else 0,

    "total_predicted_players": counts["pred_players"],
    "total_predicted_referees": counts["pred_referees"],
    "total_predicted_balls": counts["pred_balls"],
}])

image_error_df = pd.DataFrame(image_rows)

overall_error_path = WORK_DIR / "swapped_error_analysis_summary.csv"
image_error_path = WORK_DIR / "swapped_image_error_analysis.csv"

overall_error_summary.to_csv(overall_error_path, index=False)
image_error_df.to_csv(image_error_path, index=False)

print("\nOverall error summary:")
display(overall_error_summary)

print("\nSaved:")
print(overall_error_path)
print(image_error_path)

# Save a checkpoint zip after custom error-analysis CSVs are written.
safe_export_outputs(stage="after_error_analysis", cleanup_heavy=False, require_model=True)


Test images: 720
Processing 0/720
Processing 100/720
Processing 200/720
Processing 300/720
Processing 400/720
Processing 500/720
Processing 600/720
Processing 700/720

Overall error summary:


,conf_threshold,gt_team_a,gt_team_b,gt_players,gt_referees,gt_balls,team_a_detection_recall,team_b_detection_recall,player_detection_recall,ball_recall,referee_recall,referee_wrong_as_player_rate,player_wrong_as_referee_rate,total_predicted_players,total_predicted_referees,total_predicted_balls
0,0.25,3115,3158,6273,787,641,0.9817,0.9804,0.981,0.5616,0.8895,0.1461,0.0016,6740,753,447



Saved:
/kaggle/working/swapped_error_analysis_summary.csv
/kaggle/working/swapped_image_error_analysis.csv
Creating export folder: /kaggle/working/checkpoint_after_error_analysis_swapped_3class_outputs
Copied best model: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960/weights/best.pt
Copied CSV: /kaggle/working/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/swapped_error_analysis_summary.csv
Copied CSV: /kaggle/working/swapped_image_error_analysis.csv
Copied plot: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960_test_eval/confusion_matrix_normalized.png
Copied plot: /kaggle/working/three_class_swapped_experiments/hf_football_

'/kaggle/working/checkpoint_after_error_analysis_swapped_3class_outputs.zip'

# Cell 8 — Threshold comparison

In [8]:
# =========================================================
# Threshold comparison for player/ball/referee
# =========================================================

THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
MIN_CONF = min(THRESHOLDS)

comparison_rows = []

# Run predictions once at lowest threshold
all_preds = []

print("Running predictions once at conf =", MIN_CONF)

for idx, img_path in enumerate(image_paths):
    if idx % 100 == 0:
        print(f"Predicting {idx}/{len(image_paths)}")

    frame = cv2.imread(str(img_path))

    if frame is None:
        continue

    h, w = frame.shape[:2]

    gt_objects = load_original_labels(
        ORIGINAL_TEST_LABELS / f"{img_path.stem}.txt",
        w,
        h
    )

    result = eval_model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=MIN_CONF,
        device=0,
        verbose=False
    )[0]

    predictions = []

    if result.boxes is not None:
        xyxy = result.boxes.xyxy.cpu().numpy()
        clss = result.boxes.cls.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()

        for box, cls_id, conf in zip(xyxy, clss, confs):
            predictions.append({
                "box": [float(x) for x in box],
                "cls": int(cls_id),
                "conf": float(conf)
            })

    all_preds.append({
        "image": img_path.name,
        "gt_objects": gt_objects,
        "predictions": predictions
    })

print("Prediction cache ready.")

for threshold in THRESHOLDS:
    c = Counter()

    for item in all_preds:
        gt_objects = item["gt_objects"]
        predictions = [p for p in item["predictions"] if p["conf"] >= threshold]

        gt_players = [g for g in gt_objects if g["type"] == "player"]
        gt_team_a = [g for g in gt_players if g["team"] == 0]
        gt_team_b = [g for g in gt_players if g["team"] == 1]
        gt_refs = [g for g in gt_objects if g["type"] == "referee"]
        gt_balls = [g for g in gt_objects if g["type"] == "ball"]

        pred_players = [p for p in predictions if p["cls"] == 0]
        pred_balls = [p for p in predictions if p["cls"] == 1]
        pred_refs = [p for p in predictions if p["cls"] == 2]

        c["gt_team_a"] += len(gt_team_a)
        c["gt_team_b"] += len(gt_team_b)
        c["gt_players"] += len(gt_players)
        c["gt_referees"] += len(gt_refs)
        c["gt_balls"] += len(gt_balls)

        c["pred_players"] += len(pred_players)
        c["pred_referees"] += len(pred_refs)
        c["pred_balls"] += len(pred_balls)

        c["matched_team_a_as_player"] += len(greedy_match(pred_players, gt_team_a, IOU_THRESHOLD))
        c["matched_team_b_as_player"] += len(greedy_match(pred_players, gt_team_b, IOU_THRESHOLD))
        c["matched_balls"] += len(greedy_match(pred_balls, gt_balls, IOU_THRESHOLD))
        c["matched_referee_as_referee"] += len(greedy_match(pred_refs, gt_refs, IOU_THRESHOLD))
        c["referee_wrongly_as_player"] += len(greedy_match(pred_players, gt_refs, IOU_THRESHOLD))
        c["player_wrongly_as_referee"] += len(greedy_match(pred_refs, gt_players, IOU_THRESHOLD))

    comparison_rows.append({
        "threshold": threshold,
        "team_a_detection_recall": round(c["matched_team_a_as_player"] / c["gt_team_a"], 4) if c["gt_team_a"] > 0 else 0,
        "team_b_detection_recall": round(c["matched_team_b_as_player"] / c["gt_team_b"], 4) if c["gt_team_b"] > 0 else 0,
        "player_detection_recall": round((c["matched_team_a_as_player"] + c["matched_team_b_as_player"]) / c["gt_players"], 4) if c["gt_players"] > 0 else 0,
        "ball_recall": round(c["matched_balls"] / c["gt_balls"], 4) if c["gt_balls"] > 0 else 0,
        "referee_recall": round(c["matched_referee_as_referee"] / c["gt_referees"], 4) if c["gt_referees"] > 0 else 0,
        "referee_wrong_as_player_rate": round(c["referee_wrongly_as_player"] / c["gt_referees"], 4) if c["gt_referees"] > 0 else 0,
        "player_wrong_as_referee_rate": round(c["player_wrongly_as_referee"] / c["gt_players"], 4) if c["gt_players"] > 0 else 0,
        "total_predicted_players": c["pred_players"],
        "total_predicted_referees": c["pred_referees"],
        "total_predicted_balls": c["pred_balls"],
    })

threshold_df = pd.DataFrame(comparison_rows)

threshold_path = WORK_DIR / "swapped_threshold_comparison.csv"
threshold_df.to_csv(threshold_path, index=False)

display(threshold_df)

print("Saved:", threshold_path)

# Save a checkpoint zip after threshold-comparison CSV is written.
safe_export_outputs(stage="after_threshold_comparison", cleanup_heavy=False, require_model=True)


Running predictions once at conf = 0.1
Predicting 0/720
Predicting 100/720
Predicting 200/720
Predicting 300/720
Predicting 400/720
Predicting 500/720
Predicting 600/720
Predicting 700/720
Prediction cache ready.


,threshold,team_a_detection_recall,team_b_detection_recall,player_detection_recall,ball_recall,referee_recall,referee_wrong_as_player_rate,player_wrong_as_referee_rate,total_predicted_players,total_predicted_referees,total_predicted_balls
0,0.10,0.9865,0.9864,0.9864,0.6802,0.8958,0.2046,0.0016,7540,787,741
1,0.15,0.9856,0.9845,0.9850,0.6334,0.8920,0.1842,0.0016,7134,771,608
2,0.20,0.9823,0.9823,0.9823,0.5944,0.8907,0.1665,0.0016,6901,759,516
3,0.25,0.9817,0.9804,0.9810,0.5616,0.8895,0.1461,0.0016,6740,753,447
4,0.30,0.9801,0.9785,0.9793,0.4977,0.8856,0.1296,0.0016,6607,746,382
5,0.40,0.9775,0.9740,0.9758,0.3136,0.8666,0.1017,0.0013,6397,723,240


Saved: /kaggle/working/swapped_threshold_comparison.csv
Creating export folder: /kaggle/working/checkpoint_after_threshold_comparison_swapped_3class_outputs
Copied best model: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960/weights/best.pt
Copied CSV: /kaggle/working/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/checkpoint_after_error_analysis_swapped_3class_outputs/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/checkpoint_after_error_analysis_swapped_3class_outputs/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/swapped_error_analysis_summary.csv
Copied CSV: /kaggle/working/checkpoint_after_error_analysis_swapped_3class_outpu

'/kaggle/working/checkpoint_after_threshold_comparison_swapped_3class_outputs.zip'

# Cell 9 — Zip outputs

In [9]:
from pathlib import Path
import shutil
import os
import gc

# =========================================================
# Final zip/export cell
# Robust against reruns, but cannot recover files after Kaggle deletes /kaggle/working.
# If this cell says no model exists, rerun all cells from the top in one Kaggle session.
# =========================================================

zip_path = safe_export_outputs(stage="final", cleanup_heavy=True, require_model=True)

print("\nDone. Download this file from Kaggle output:")
print(zip_path)


Creating export folder: /kaggle/working/final_swapped_3class_outputs
Copied best model: /kaggle/working/three_class_swapped_experiments/hf_football_3cls_swapped_960/weights/best.pt
Copied CSV: /kaggle/working/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/checkpoint_after_error_analysis_swapped_3class_outputs/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/checkpoint_after_threshold_comparison_swapped_3class_outputs/three_class_swapped_per_class.csv
Copied CSV: /kaggle/working/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/checkpoint_after_test_validation_swapped_3class_outputs/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/checkpoint_after_error_analysis_swapped_3class_outputs/three_class_swapped_summary.csv
Copied CSV: /kaggle/working/checkpoint_after_threshold_comparison_swapped_3class_outputs/three_class_swapped